# AI Powered Mock Interview System

This notebook implements a backend-ready logic for an AI mockup interview system.

**Features:**
- Resume Parsing (PDF)
- Skill Extraction (Hybrid: NLP + Keywords)
- Question Generation using Google Gemini API
- Interview Simulation with Timers

**Requirements:**
- Python 3.8+
- Google Gemini API Key (set as `GEMINI_API_KEY` env var)

In [ ]:
# Install necessary libraries
# !pip install google-generativeai pdfplumber spacy reportlab
# !python -m spacy download en_core_web_sm

In [ ]:
import os
import json
import time
import re
import pdfplumber
import spacy
import google.generativeai as genai
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

# Load spaCy model
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Downloading spaCy model...")
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

# Predefined Skill Keywords (Extensible)
SKILL_KEYWORDS = {
    "python", "java", "c++", "javascript", "typescript", "react", "angular", "vue",
    "node.js", "django", "flask", "fastapi", "sql", "nosql", "mongodb", "postgresql",
    "aws", "azure", "gcp", "docker", "kubernetes", "git", "machine learning",
    "deep learning", "nlp", "tensorflow", "pytorch", "scikit-learn", "pandas", "numpy",
    "html", "css", "sass", "rest api", "graphql", "ci/cd", "agile", "scrum", "linux"
}

## 1. Resume Parsing Logic

In [ ]:
def extract_resume_text(file_path):
    """
    Extracts text from a PDF resume using pdfplumber.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Resume file not found at: {file_path}")

    text = ""
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                extracted = page.extract_text()
                if extracted:
                    text += extracted + "\n"
    except Exception as e:
        raise RuntimeError(f"Error parsing PDF: {str(e)}")
    
    # clean text
    clean_text = re.sub(r'\s+', ' ', text).strip()
    return clean_text

## 2. Skill Extraction

In [ ]:
def extract_skills(text):
    """
    Extracts skills from text using a hybrid approach:
    1. Direct keyword matching from a predefined set.
    2. spaCy NER to identify Organizations/Products (potential tech).
    """
    text_lower = text.lower()
    found_skills = set()

    # 1. Keyword Matching
    for skill in SKILL_KEYWORDS:
        # Use regex to find standalone words to avoid overly matching substrings
        # e.g. "java" in "javascript" (though SKILL_KEYWORDS has both, handling boundaries is safer)
        if re.search(r'\b' + re.escape(skill) + r'\b', text_lower):
            found_skills.add(skill)

    # 2. Basic NLP Extraction (Optional enhancement)
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ in {"ORG", "PRODUCT", "WORK_OF_ART"}:
            candidate = ent.text.lower().strip()
            if candidate in SKILL_KEYWORDS:
                 found_skills.add(candidate)

    return list(found_skills)

## 3. Gemini API Integration

In [ ]:
def build_gemini_prompt(skills, position, yoe, total_questions):
    """
    Constructs a structured prompt for Gemini to generate JSON Interview Questions.
    """
    technical_count = int(total_questions * 0.6)
    project_count = int(total_questions * 0.25)
    behavioral_count = total_questions - technical_count - project_count

    prompt = f"""
    You are a Senior Technical Interviewer. Generate a mock interview based on the following candidate profile:

    **Candidate Profile:**
    - Position: {position}
    - Years of Experience: {yoe}
    - Skills: {', '.join(skills)}

    **Task:**
    Generate exactly {total_questions} interview questions categorized as follows:
    1. Technical ({technical_count} questions): Based on extracting skills and role requirements. Difficulty must match {yoe} YOE.
    2. Project/Practical ({project_count} questions): Scenario-based questions applying their skills.
    3. Behavioral ({behavioral_count} questions): Cultural fit and soft skills.

    **Strict Constraints:**
    - Output MUST be valid pure JSON.
    - Do not include markdown formatting like ```json ... ```.
    - Do not include any preamble or explanation.
    - Questions must be relevant to the specific list of skills provided.

    **Output Format:**
    {{
      "position": "{position}",
      "yoe": {yoe},
      "total_questions": {total_questions},
      "timer_per_section": 15,
      "questions": {{
        "technical": ["q1", "q2", ...],
        "project": ["q1", "q2", ...],
        "behavioral": ["q1", "q2", ...]
      }}
    }}
    """
    return prompt

def generate_questions_with_gemini(prompt):
    """
    Calls Gemini API and returns the parsed JSON response.
    """
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
         # Placeholder for demo purposes if key is missing, or raise error
         raise ValueError("GEMINI_API_KEY environment variable not set.")

    genai.configure(api_key=api_key)
    model = genai.GenerativeModel('gemini-pro')

    try:
        response = model.generate_content(prompt)
        response_text = response.text.strip()
        
        # Cleanup if model adds markdown blocks
        if response_text.startswith("```json"):
            response_text = response_text[7:]
        if response_text.endswith("```"):
            response_text = response_text[:-3]
            
        return json.loads(response_text)
    except Exception as e:
        print(f"Gemini API Error: {e}")
        return None

## 4. Interview Simulation

In [ ]:
def run_mock_interview(config):
    """
    Simulates the interview process.
    """
    print("\n" + "="*40)
    print(f"STARTING MOCK INTERVIEW FOR: {config['position'].upper()}")
    print("="*40)
    
    interview_data = config['interview_data']
    if not interview_data:
        print("Failed to generate valid interview questions.")
        return

    sections = interview_data.get("questions", {})
    minutes_per_section = config.get("time_per_section", 5)

    for section_name, questions in sections.items():
        print(f"\n>>> SECTION: {section_name.upper()}")
        print(f"Time Alotted: {minutes_per_section} minutes")
        
        # Simulate timer (non-blocking logic for demo)
        # In a real app, this would be handled by the frontend.
        start_time = time.time()
        
        for i, q in enumerate(questions, 1):
            print(f"Q{i}: {q}")
            # Simulation delay to reading/thinking
            time.sleep(0.5) 
        
        elapsed = time.time() - start_time
        print(f"\n(Section completed in {elapsed:.2f} seconds simulation)")
        print("-"*20)

    print("\n" + "="*40)
    print("INTERVIEW COMPLETE")
    print("="*40)

## 5. Demo / Main Execution

In [ ]:
def create_dummy_resume(filename="sample_resume.pdf"):
    """
    Creates a dummy PDF resume for testing purposes.
    """
    c = canvas.Canvas(filename, pagesize=letter)
    c.drawString(100, 750, "John Doe")
    c.drawString(100, 730, "Software Engineer")
    c.drawString(100, 700, "Experience: 3 Years")
    c.drawString(100, 680, "Skills: Python, React, AWS, Docker, Machine Learning, SQL")
    c.drawString(100, 660, "Projects: Built a chatbot using NLP and FastAPI.")
    c.save()
    print(f"Created dummy resume: {filename}")
    return filename

# --- Main Execution ---
if __name__ == "__main__":
    # 1. Setup Demo File
    resume_file = create_dummy_resume()
    
    # 2. Extract Data
    print("\n[1] Extracting Resume Text...")
    raw_text = extract_resume_text(resume_file)
    
    print("[2] Extracting Skills...")
    skills = extract_skills(raw_text)
    print(f"Found Skills: {skills}")

    # 3. User Configuration (Mock Inputs)
    config = {
        "position": "Full Stack Developer",
        "yoe": 3,
        "total_questions": 10,
        "time_per_section": 10  # minutes
    }

    # 4. Generate Questions
    print("\n[3] Generating Questions via Gemini...")
    try:
        prompt = build_gemini_prompt(skills, config['position'], config['yoe'], config['total_questions'])
        # NOTE: Ensure GEMINI_API_KEY is set in your environment variables for this to work!
        # os.environ["GEMINI_API_KEY"] = "YOUR_ACTUAL_KEY_HERE"
        
        questions_data = generate_questions_with_gemini(prompt)
        
        if questions_data:
            config['interview_data'] = questions_data
            
            # 5. Run Interview
            run_mock_interview(config)
        else:
            print("WARNING: Could not generate questions. Check API Key.")
            
    except ValueError as ve:
        print(f"Setup Error: {ve}")
    except Exception as e:
        print(f"Unexpected Error: {e}")